# Visual Health — qualité sur DDR (Kaggle)

Embeddings DINOv2-B « qualité » des **13 673 images de gradation DDR**, y compris les
1 151 images que les lecteurs n'ont pas pu grader (stade 5). Ils servent à valider, sur une
base indépendante, le détecteur de qualité proposé au tuteur (appris le 10 novembre sur
DeepDRiD). Plan fixé avant tout calcul : `scripts/quality_ddr_validation.py`. L'analyse se
fait ensuite en local, sur processeur.

Le notebook s'arrête si le dataset diffère de l'archive officielle des auteurs, ou si les
poids DINOv2-B téléchargés n'ont pas exactement les mêmes valeurs que ceux du détecteur.

**Avant de lancer** :
1. `Add Input` → onglet **Your Datasets** → le dataset privé DDR (`marco865/ddr-grading-officiels`).
2. `Settings > Accelerator > None` (processeur : ne consomme pas le quota GPU),
   `Settings > Internet > On` (poids DINOv2-B via timm).
3. **`Save Version > Save & Run All (Commit)`** (~1 à 2 h sur processeur). Le fichier
   `visual_health_quality_ddr_embeddings.zip` apparaît ensuite dans l'onglet `Output`.


In [ ]:
!nproc
!df -h /kaggle/working /kaggle/temp | tail -2


In [ ]:
import os
import subprocess

OUTPUT = "/kaggle/working"  # seul le zip y est écrit
WORK = "/kaggle/temp"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

REPO = "sumarc183-design/visual-health-platform"
REVISION = "main"  # ou un commit précis, pour rejouer un run à l'identique


def _git(*args):
    if subprocess.run(["git", *args], capture_output=True).returncode != 0:
        raise RuntimeError(f"git {args[0]} a échoué")


if not os.path.exists("visual-health-platform"):
    _git("clone", "-q", f"https://github.com/{REPO}.git", "visual-health-platform")
_git("-C", "visual-health-platform", "fetch", "-q", "origin")
_local = subprocess.run(
    ["git", "-C", "visual-health-platform", "status", "--porcelain", "-uno"], capture_output=True, text=True
).stdout.strip()
if _local:  # ne pas écraser des modifications faites dans la copie (session réutilisée)
    raise RuntimeError(f"modifications locales dans visual-health-platform, à sauvegarder avant reset :\n{_local}")
_git("-C", "visual-health-platform", "reset", "-q", "--hard", "origin/main" if REVISION == "main" else REVISION)
!cd visual-health-platform && git log --oneline -1
%cd {WORK}/visual-health-platform
import subprocess, sys  # noqa: E401
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)  # arrêt si échec


## Vérification du dataset

Empreinte de l'inventaire officiel (fichiers de stades et 13 673 images), calculée sur
l'archive des auteurs : `inventory_digest` dans `src/data/ddr_dataset.py`.


In [ ]:
import glob

hits = sorted(glob.glob("/kaggle/input/**/DR_grading/train.txt", recursive=True))
if not hits:
    raise RuntimeError(f"DDR introuvable : ajouter le dataset privé en input. Inputs : {os.listdir('/kaggle/input')}")
DDR = os.path.dirname(os.path.dirname(hits[0]))
print("DDR :", DDR)

from src.data.ddr_dataset import inventory_digest

EXPECTED = "b05f6d1711402a1e1a77f5dd0e0a7f32741d6780dd604b18533c482d7611cfcd"
digest, n = inventory_digest(DDR)
print(n, "fichiers, empreinte", digest)
if digest != EXPECTED:
    raise RuntimeError("Le dataset DDR diffère de l'archive officielle : arrêt avant tout calcul")
print("Dataset identique à l'archive officielle")


## Poids DINOv2-B

Téléchargés par timm, puis comparés à ceux du détecteur par l'empreinte de leurs tenseurs
(`df4a90fee93f84ed`, indépendante de la sérialisation du fichier).


In [ ]:
import timm
import torch

from scripts.quality_ddr_embeddings import EXPECTED_TENSOR_DIGEST, tensor_digest
from src.models.timm_backbones import DINOV2_MODEL

WEIGHTS = "third_party/dinov2/dinov2_vitb14_timm.pth"
os.makedirs(os.path.dirname(WEIGHTS), exist_ok=True)
if not os.path.exists(WEIGHTS):
    torch.save(timm.create_model(DINOV2_MODEL, pretrained=True).state_dict(), WEIGHTS)
digest = tensor_digest(WEIGHTS)
print("Empreinte des tenseurs :", digest)
if digest != EXPECTED_TENSOR_DIGEST:
    raise RuntimeError("poids DINOv2-B différents de ceux du détecteur : arrêt avant tout calcul")


## Embeddings


In [ ]:
import time

OUTS = ("embeddings/quality_ddr_dinov2_vitb14.npz", "embeddings/quality_ddr_labels.csv")
t0 = time.time()
!python -u scripts/quality_ddr_embeddings.py --ddr-root "{DDR}" --checkpoint {WEIGHTS} --threads {os.cpu_count()} 2>&1 | grep -v -i warning
for out in OUTS:
    if not os.path.exists(out) or os.path.getmtime(out) < t0:
        raise RuntimeError(f"{out} absent ou ancien : quality_ddr_embeddings.py a échoué (voir ci-dessus)")
print(f"{(time.time() - t0) / 60:.0f} min")


## Garder le résultat


In [ ]:
import zipfile

archive = f"{OUTPUT}/visual_health_quality_ddr_embeddings.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in OUTS:
        z.write(f)
subprocess.run(["python", "scripts/run_manifest.py", "--archive", archive, "--requested-revision", REVISION], check=True)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo)")
